### Lecture 8: Let's build the GPT Tokenizer

[Youtube video lecture](https://www.youtube.com/watch?v=zduSFxRajkE)  
[code for this video](https://colab.research.google.com/drive/1y0KnCFZvGVf_odSfcNAws6kcDD7HsI0L?usp=sharing)  
[minbpe repo](https://github.com/karpathy/minbpe)  

Video upload date: Feb 21, 2024 (length: 2 hr 13 min)  
Watched on: Nov 30, 2025  
Reproduced on: Nov 30, 2025  

Content:
- tokenization and tokenizer

Resource:  
- [tiktokenizer demonstration website](https://tiktokenizer.vercel.app/)
- [GPT2 opensource repo (encoder)](https://github.com/openai/gpt-2/blob/master/src/encoder.py)

---

## Q: Why are there only 256 “byte symbols”? Is that enough to represent all characters (English, Chinese, emoji, etc.)?

### Short answer

- A **byte** has 256 possible values (0–255).  
- Modern **byte-level BPE** tokenizers start from **256 byte symbols**, but:
  - They **do not** say “there are only 256 characters”.
  - They say “we treat each **byte** as a basic symbol”.
- Any Unicode character (including emoji) can be represented as **one or more bytes** (UTF-8), so 256 byte values are enough to represent arbitrarily many characters.
- BPE then builds a **much larger vocabulary** (e.g., 50k tokens) by merging frequently occurring byte sequences.

---

## 1. Bytes vs Characters

- **Byte**: 8 bits → 2⁸ = **256** possible values (0–255).
- **Character** (Unicode code point): abstract symbol like `A`, `你`, `😊`.

In **UTF-8**, a character is encoded as **1–4 bytes**.

Examples (UTF-8, hex):

| Character | UTF-8 bytes (hex) | # of bytes |
|----------:|-------------------|-----------:|
| `A`       | 41                | 1          |
| `é`       | C3 A9             | 2          |
| `你`      | E4 BD A0          | 3          |
| `😊`      | F0 9F 98 8A       | 4          |

Even though there are only 256 possible byte values, you can represent **many characters** by combining bytes into sequences.

**Analogy**

- 26 letters → infinitely many words.  
- 256 byte values → effectively unlimited characters.

---

## 2. Byte-Level BPE: What “256 symbols” really means

Byte-level BPE (e.g., GPT-2 tokenizer):

1. **Initial alphabet**:  
   - 256 symbols, one for each byte value `0–255`.
2. Convert text → UTF-8 bytes.
3. Run BPE:
   - Count frequent **adjacent byte pairs**.
   - Merge the most frequent pair into a new token.
   - Repeat many times.

After many merges:

- Final vocabulary size is often **tens of thousands** (e.g., 50k).
- Tokens correspond to:
  - common substrings: `"the"`, `"ing"`, `"tion"`, …
  - single multi-byte characters: `"你"`, `"的"`, `"😊"`, …
  - other frequent byte sequences.

So:

- The model’s **token IDs** are *not* limited to 256.
- Only the **base building blocks** (bytes) are 256.


## 3. Python Examples

In [1]:
text = "Aé你😊"
for ch in text:
    utf8_bytes = ch.encode("utf-8")
    byte_vals = list(utf8_bytes)
    hex_vals = [hex(b) for b in byte_vals]
    print(
        f"Character: {ch!r} | "
        f"code point: U+{ord(ch):04X} | "
        f"bytes (dec): {byte_vals} | "
        f"bytes (hex): {hex_vals}"
    )

Character: 'A' | code point: U+0041 | bytes (dec): [65] | bytes (hex): ['0x41']
Character: 'é' | code point: U+00E9 | bytes (dec): [195, 169] | bytes (hex): ['0xc3', '0xa9']
Character: '你' | code point: U+4F60 | bytes (dec): [228, 189, 160] | bytes (hex): ['0xe4', '0xbd', '0xa0']
Character: '😊' | code point: U+1F60A | bytes (dec): [240, 159, 152, 138] | bytes (hex): ['0xf0', '0x9f', '0x98', '0x8a']


This shows: same string length in Python, different byte lengths in UTF-8.

### Toy “BPE idea” over bytes

In [2]:
from collections import Counter

# Example text
text = "helloooo 😊😊"
# Encode to UTF-8 bytes
data = text.encode("utf-8")
print("Raw bytes:", list(data))

# Step 1: treat each byte as a token (initial alphabet: 0–255)
tokens = list(data)  # just ints 0–255

# Step 2: find frequent adjacent pairs
pairs = Counter()
for i in range(len(tokens) - 1):
    pair = (tokens[i], tokens[i + 1])
    pairs[pair] += 1

print("Most common byte pairs (toy):")
for pair, count in pairs.most_common(5):
    print(pair, "-> count:", count)

# In real BPE:
# - You'd merge the most frequent pair into a new token ID,
# - Replace those occurrences,
# - Recompute pairs and repeat many times,
# - Ending with a much larger vocabulary than 256.

Raw bytes: [104, 101, 108, 108, 111, 111, 111, 111, 32, 240, 159, 152, 138, 240, 159, 152, 138]
Most common byte pairs (toy):
(111, 111) -> count: 3
(240, 159) -> count: 2
(159, 152) -> count: 2
(152, 138) -> count: 2
(104, 101) -> count: 1


---
## Karparthy content

In [3]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience
print('---')
print(text)
print("length:", len(text))
print('---')
print(tokens)
print("length:", len(tokens))

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

In [4]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]): # Pythonic way to iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
# print(stats)
print(sorted(((v,k) for k,v in stats.items()), reverse=True))

[(20, (101, 32)), (15, (240, 159)), (12, (226, 128)), (12, (105, 110)), (10, (115, 32)), (10, (97, 110)), (10, (32, 97)), (9, (32, 116)), (8, (116, 104)), (7, (159, 135)), (7, (159, 133)), (7, (97, 114)), (6, (239, 189)), (6, (140, 240)), (6, (128, 140)), (6, (116, 32)), (6, (114, 32)), (6, (111, 114)), (6, (110, 103)), (6, (110, 100)), (6, (109, 101)), (6, (104, 101)), (6, (101, 114)), (6, (32, 105)), (5, (117, 115)), (5, (115, 116)), (5, (110, 32)), (5, (100, 101)), (5, (44, 32)), (5, (32, 115)), (4, (116, 105)), (4, (116, 101)), (4, (115, 44)), (4, (114, 105)), (4, (111, 117)), (4, (111, 100)), (4, (110, 116)), (4, (110, 105)), (4, (105, 99)), (4, (104, 97)), (4, (103, 32)), (4, (101, 97)), (4, (100, 32)), (4, (99, 111)), (4, (97, 109)), (4, (85, 110)), (4, (32, 119)), (4, (32, 111)), (4, (32, 102)), (4, (32, 85)), (3, (118, 101)), (3, (116, 115)), (3, (116, 114)), (3, (116, 111)), (3, (114, 116)), (3, (114, 115)), (3, (114, 101)), (3, (111, 102)), (3, (111, 32)), (3, (108, 108)), (

In [5]:
top_pair = max(stats, key=stats.get)
top_pair

(101, 32)

In [6]:
def merge(ids, pair, idx):
  # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
  newids = []
  i = 0
  while i < len(ids):
    # if we are not at the very last position AND the pair matches, replace it
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

tokens2 = merge(tokens, top_pair, 256)
print(tokens2)
print("length:", len(tokens2))

[5, 6, 99, 9, 1]
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140, 240, 159, 135, 169, 226, 128, 140, 240, 159, 135, 170, 33, 32, 240, 159, 152, 132, 32, 84, 104, 256, 118, 101, 114, 121, 32, 110, 97, 109, 256, 115, 116, 114, 105, 107, 101, 115, 32, 102, 101, 97, 114, 32, 97, 110, 100, 32, 97, 119, 256, 105, 110, 116, 111, 32, 116, 104, 256, 104, 101, 97, 114, 116, 115, 32, 111, 102, 32, 112, 114, 111, 103, 114, 97, 109, 109, 101, 114, 115, 32, 119, 111, 114, 108, 100, 119, 105, 100, 101, 46, 32, 87, 256, 97, 108, 108, 32, 107, 110, 111, 119, 32, 119, 256, 111, 117, 103, 104, 116, 32, 116, 111, 

In [7]:
vocab_size = 276 # the desired final vocabulary size
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (101, 32) into a new token 256
merging (240, 159) into a new token 257
merging (226, 128) into a new token 258
merging (105, 110) into a new token 259
merging (115, 32) into a new token 260
merging (97, 110) into a new token 261
merging (116, 104) into a new token 262
merging (257, 133) into a new token 263
merging (257, 135) into a new token 264
merging (97, 114) into a new token 265
merging (239, 189) into a new token 266
merging (258, 140) into a new token 267
merging (267, 264) into a new token 268
merging (101, 114) into a new token 269
merging (111, 114) into a new token 270
merging (116, 32) into a new token 271
merging (259, 103) into a new token 272
merging (115, 116) into a new token 273
merging (261, 100) into a new token 274
merging (32, 262) into a new token 275


In [8]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 616
ids length: 451
compression ratio: 1.37X


### decoding

Given a sequence of integers in the range [0, vocab_size], what is the text?


In [9]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
  # given ids (list of integers), return Python string
  tokens = b"".join(vocab[idx] for idx in ids)
  text = tokens.decode("utf-8", errors="replace")
  return text

print(decode([128]))

�


### Forced splits using regex patterns (GPT series)


In [10]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']


In [11]:
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else:
        print(i)
"""
print(re.findall(gpt2pat, example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', ':', '\n       ', ' print', '(', 'i', ')', '\n']


In [12]:
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


## What is SentencePiece? What does it have to do with BPE?

### 1. What is SentencePiece?

**SentencePiece** is a **tokenizer toolkit** (by Google) for training and using subword tokenizers.  
It’s not a single algorithm; it’s a **framework** that supports multiple algorithms, mainly:

- **Unigram Language Model** (default in many SentencePiece setups)
- **BPE (Byte Pair Encoding)**

Key features:

- Works **directly on raw text** (no need for pre-tokenized “words”):
  - It treats input as a **sequence of Unicode characters** or bytes.
  - Useful for languages without spaces (Chinese, Japanese, etc.).
- Builds a subword vocabulary (like BPE) of a given size (e.g. 32k).
- Produces a self-contained `.model` file (vocab + rules) that can be used in many environments.

Many modern models (e.g., older T5, ALBERT, etc.) use SentencePiece-based tokenizers.

---

### 2. How is SentencePiece related to BPE?

Think of it this way:

- **BPE** = one **specific subword algorithm** (originally a compression technique) adapted for tokenization.
- **SentencePiece** = a **tool/library** that can **implement BPE** (and other algorithms).

So:

- BPE is like a “recipe”.
- SentencePiece is like a “kitchen” where you can cook with:
  - the BPE recipe,
  - or the Unigram recipe,
  - etc.

When you see “SentencePiece BPE”, it means:

> “We used SentencePiece to train a tokenizer, using BPE as the underlying algorithm.”

When you see “SentencePiece Unigram”, it means:

> “We used SentencePiece to train a tokenizer, using the Unigram LM algorithm.”

---

### 3. SentencePiece vs “Classic” BPE Implementations

**Classic BPE tokenization** (like early GPT-2 style):

- Often starts from **byte-level** or character-level symbols.
- Repeatedly merges **most frequent pairs**.
- Implementation is usually custom or via specific libraries (e.g., Hugging Face `tokenizers`).

**SentencePiece with BPE**:

- Same core idea (merge frequent pairs), but:
  - Handles **Unicode normalization**, whitespace handling, etc., in a standardized way.
  - Can treat the entire input as a raw stream (no pre-tokenization).
  - Generates a portable model file (`.model` + `.vocab`).

So BPE inside SentencePiece is a **more standardized, production-ready implementation** of BPE tokenization.

---

### 4. SentencePiece Algorithms (High Level)

#### 4.1. BPE in SentencePiece

- Very similar to the BPE you’ve already learned:
  - Start from characters (or bytes),
  - Merge most frequent pairs,
  - Repeat until vocab size is reached.
- The final tokens are subword units; common patterns become single tokens.

#### 4.2. Unigram LM (the other main algorithm)

- Starts with a **large initial vocabulary** of candidate subwords.
- Trains a **probabilistic model** (unigram LM) over subwords.
- Iteratively **removes** subwords that do not help the likelihood much.
- At the end, you get a compact, probabilistic subword vocabulary.

Unigram vs BPE:

- BPE: merge-based, deterministic pair merging.
- Unigram: probability-based, candidate removal; often more flexible in segmentation.

SentencePiece can do **both**, depending on the `--model_type` you choose.

---

### 5. Why do people use SentencePiece?

Some reasons:

1. **Language-agnostic**:
   - Works well for English, Chinese, Japanese, etc.
   - No need for language-specific word segmentation beforehand.

2. **Reproducible**:
   - Training config + `.model` file = you can reproduce exactly the same tokenizer behavior.

3. **Integration**:
   - Easily used in C++, Python, Java, etc.
   - Widely adopted in Google and some open-source models.

4. **Flexibility**:
   - Can choose BPE or Unigram.
   - Can configure vocabulary size, special tokens, normalization rules, etc.

---

### 6. Tiny Example (Python, conceptual)

Below is a **conceptual** example using the `sentencepiece` Python package:

In [14]:
import sentencepiece as spm

# 1. Train a SentencePiece model with BPE
spm.SentencePieceTrainer.Train(
    input="data/tinyshakespeare.txt",           # your raw text file
    model_prefix="my_bpe",
    vocab_size=8000,
    model_type="bpe"              # <-- use BPE; "unigram" is the other main option
)

# This creates:
# - my_bpe.model
# - my_bpe.vocab

# 2. Use the trained tokenizer
sp = spm.SentencePieceProcessor()
sp.load("my_bpe.model")

text = "Hello world! 你好，世界！"

ids = sp.encode(text, out_type=int)
pieces = sp.encode(text, out_type=str)

print("Pieces:", pieces)
print("IDs:", ids)

# Decode back
print("Decoded:", sp.decode(ids))

Pieces: ['▁H', 'ello', '▁world', '!', '▁', '你好', ',', '世界', '!']
IDs: [73, 3573, 624, 7986, 7941, 0, 7956, 0, 7986]
Decoded: Hello world!  ⁇ , ⁇ !


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: data/tinyshakespeare.txt
  input_format: 
  model_prefix: my_bpe
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
 